# E040 — Logistic Regression Regularization Sweep (Image)

**Motivation:** E033 uses LogReg C=1.0 (default). Regularization strength (C) and type (L1/L2/elastic) may affect performance, especially with adversarial training data.

**Hypothesis:** L2 regularization with tuned C will outperform default C=1.0. L1 (sparse) or elastic net may help with high-dimensional PCA features.

**Configs:**
- `C=0.1`: Stronger regularization
- `C=1.0`: E033 baseline
- `C=10.0`: Weaker regularization
- `C=100.0`: Very weak (near unregularized)
- `L1 C=1.0`: L1 penalty (sparse)

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('../src').resolve()))

import numpy as np
from PIL import Image
from scipy.ndimage import rotate
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf

SEED = 67
N_PCA = 50

DATA = Path('../data').resolve()
manifest = load_manifest(DATA)
y_all = manifest['label'].to_numpy()
print(f'{len(manifest)} samples')

E033_REF = {'mean_eer': 0.51, 'std_eer': 0.36}

222 samples


In [2]:
def _find_png(stem, data_dir):
    for sf in ("target_train", "target_dev", "non_target_train", "non_target_dev"):
        p = data_dir / sf / (stem + ".png")
        if p.exists(): return p
    raise FileNotFoundError(stem)

def _load_image(path):
    img = Image.open(path).convert("RGB")
    if img.size != (80, 80):
        img = img.resize((80, 80), Image.BILINEAR)
    return np.array(img, dtype=np.float32).mean(axis=2)

def train_image_model(df, data_dir, reg_config, seed):
    """Train image model with specified regularization."""
    rng = np.random.default_rng(seed)
    C, penalty = reg_config
    
    X, y = [], []
    for _, row in df.iterrows():
        orig = _load_image(_find_png(row["stem"], data_dir))
        vecs = [orig, orig[:, ::-1].copy()]
        vecs += [np.clip(orig * rng.uniform(0.7, 1.3), 0, 255)]
        vecs += [np.clip(orig + rng.normal(0, 15, orig.shape), 0, 255)]
        
        # Adversarial rotation for target
        if row["label"] == 1:
            for angle in [-10, -5, 5, 10]:
                vecs.append(rotate(orig, angle, reshape=False, order=1, mode='constant', cval=0))
        
        for v in vecs:
            X.append(v.flatten()); y.append(row["label"])
    
    X = np.array(X)
    scaler = StandardScaler()
    pca = PCA(n_components=N_PCA, random_state=SEED)
    
    solver = 'saga' if penalty == 'l1' else 'lbfgs'
    clf = LogisticRegression(C=C, penalty=penalty, solver=solver, max_iter=2000, random_state=SEED)
    
    X_pca = pca.fit_transform(scaler.fit_transform(X))
    clf.fit(X_pca, y)
    
    return scaler, pca, clf

def score_image(png_path, scaler, pca, clf):
    x = _load_image(png_path).flatten().reshape(1, -1)
    x_pca = pca.transform(scaler.transform(x))
    return float(clf.decision_function(x_pca)[0])

print('Functions defined')

Functions defined


## 2. Cross-validation

In [3]:
configs = {
    'C=0.1': (0.1, 'l2'),
    'C=1.0 (E033)': (1.0, 'l2'),
    'C=10.0': (10.0, 'l2'),
    'C=100.0': (100.0, 'l2'),
    'L1 C=1.0': (1.0, 'l1'),
}

results = {}

for name, reg_config in configs.items():
    print(f"\n=== {name} ===")
    fold_eers = []
    
    for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
        seed_f = SEED + fold_id
        train_df = manifest.loc[train_idx]
        val_df = manifest.loc[val_idx]
        
        scaler, pca, clf = train_image_model(train_df, DATA, reg_config, seed_f)
        
        scores = []
        for _, row in val_df.iterrows():
            score = score_image(_find_png(row["stem"], DATA), scaler, pca, clf)
            scores.append(score)
        
        scores = np.array(scores)
        eer, _ = compute_eer(scores[y_all[val_idx] == 1], scores[y_all[val_idx] == 0])
        fold_eers.append(eer * 100)
        print(f"  Fold {fold_id}: EER={eer*100:.2f}%")
    
    results[name] = {
        'fold_eers': fold_eers,
        'mean': np.mean(fold_eers),
        'std': np.std(fold_eers),
    }
    print(f"  Mean ± Std: {np.mean(fold_eers):.2f} ± {np.std(fold_eers):.2f}%")

print("\n=== Summary ===")
for name, res in results.items():
    print(f"{name:15s}: {res['mean']:5.2f} ± {res['std']:5.2f}%")


=== C=0.1 ===


  Fold 0: EER=2.08%


/Users/ramsay/school/sur/project/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/ramsay/school/sur/project/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


  Fold 1: EER=0.83%


  Fold 2: EER=0.00%
  Mean ± Std: 0.97 ± 0.86%

=== C=1.0 (E033) ===


...

  Fold 0: EER=15.56%


/Users/ramsay/school/sur/project/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/ramsay/school/sur/project/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


  Fold 1: EER=25.00%


/Users/ramsay/school/sur/project/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/ramsay/school/sur/project/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


  Fold 2: EER=0.00%
  Mean ± Std: 13.52 ± 10.31%

=== Summary ===
C=0.1          :  0.97 ±  0.86%
C=1.0 (E033)   :  0.97 ±  0.86%
C=10.0         :  3.75 ±  3.92%
C=100.0        : 10.19 ±  8.17%
L1 C=1.0       : 13.52 ± 10.31%


## 3. Conclusion

Regularization effect: [TBD]
Decision: [Keep C=1.0 or adopt new value]